# EXAMPLE OF DIELECTRIC CONSTANT CALCULATOR

In [ ]:
from molify import pack, smiles2conformers
from ase.io import write

from md_runner import run_mace_polar_md
from trajectory_converter import convert_traj_to_dcd
from dielectric_calculation_mace import calculator_dielectric_constant_mace

## Create system

In [ ]:
ec = smiles2conformers("O=C1OCCO1", numConfs=5)

emc = smiles2conformers("CCOC(=O)OC", numConfs=5)

liquid_box = pack(data=[ec, emc],
                  counts=[17, 33],
                  density=1190)

write("ec_emc_initial_geometry.pdb",liquid_box)

## Run MD with MACE-POLAR

In [ ]:
run_mace_polar_md(geometry_file="initial_geometry.pdb",
                  T=300,
                  equilibration_steps=50000,
                  production_steps=2000000,
                  output_prefix="ec_emc")

## Transform trajectory to *dcd format

In [ ]:
convert_traj_to_dcd(trajectory="production.traj", topology="topology.pdb", dcd_output="trajectory.dcd")

## Calculate dielectric constant

In [ ]:
times, eps_t, eps_mean, eps_std, dipoles = calculator_dielectric_constant_mace(topology="topology.pdb",
                                                                                trajectory="trajectory.dcd",
                                                                                charges_npy="charges.npy",
                                                                                dipoles_npy="atomic_dipoles.npy",
                                                                                dump=1000,
                                                                                ts=1.0,
                                                                                T=300,
                                                                                save_plot="ec_emc_mix.png")
